In [1]:
using JSON
using DataFrames
using Statistics
using Plots
using NaNMath

In [2]:
# Load the JSON file
history = JSON.parsefile("../data/results/evolution_history.json")

println("Solution $(history["best_overall"]) found in generation $(history["generations_taken_for_optimal"])")

Solution sin((x0 / 8.67e+00)) found in generation 187


In [3]:
generations = sort([parse(Int, g) for g in keys(history["species"])])
println("Generations: $generations")

Generations: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186, 189, 192, 195, 198, 201, 204, 207, 210, 213, 216, 219, 222, 225, 228, 231, 234, 237, 240, 243, 246, 249, 252, 255, 258, 261, 264, 267, 270, 273, 276, 279, 282, 285, 288, 291, 294, 297]


In [4]:
species_id_to_idx = Dict{String,Int}()
idx_to_species_id = Dict{Int,String}()
idx_to_expression = Dict{Int,String}()
idx = 0
for gen in generations
    for species in history["species"][string(gen)]
        sid = string(species["id"])
        if !haskey(species_id_to_idx, sid)
            species_id_to_idx[sid] = idx
            idx_to_species_id[idx] = sid
            idx += 1
        end
        idx_to_expression[species_id_to_idx[sid]] = species["representative"]
    end
end
println("Unique species count: $(length(species_id_to_idx))")

Unique species count: 327


In [5]:
species_sizes_over_time = zeros(Int, length(generations), length(species_id_to_idx))
for (gi, gen) in enumerate(generations)
    for species in history["species"][string(gen)]
        sid = string(species["id"])
        species_idx = species_id_to_idx[sid]
        species_sizes_over_time[gi, species_idx] = species["size"]
    end
end

LoadError: BoundsError: attempt to access 100×327 Matrix{Int64} at index [1, 0]

In [ ]:
idxs = sortperm(species_sizes_over_time[end, :])
for idx in idxs[end-9:end]
    println("Species $(idx_to_species_id[idx]) with size $(species_sizes_over_time[end, idx]): $(idx_to_expression[idx])")
end

In [ ]:
# Heatmap of all species sizes over generations
heatmap(generations, 1:size(species_sizes_over_time, 2), species_sizes_over_time',
    xlabel="Generation", ylabel="Species Index", title="Species Size Over Generations (Heatmap)",
    color=:viridis, colorbar_title="Size", size=(800, 500))


In [ ]:
println(keys(history["species"]["0"][1]))

In [ ]:
data_over_time = fill(NaN, 4, 3, length(generations), length(species_id_to_idx))
metric_names = ["loss_raws", "complexities", "loss_insts", "loss_finals"]

# NaN-skipping median (NaNMath doesn't provide median)
nanmedian(x) = median(filter(!isnan, x))

agg_funcs = [NaNMath.minimum, nanmedian, NaNMath.maximum]
agg_labels = ["min", "median", "max"]

for (gi, gen) in enumerate(generations)
    for species in history["species"][string(gen)]
        sid = string(species["id"])
        species_idx = species_id_to_idx[sid]
        for (idx, name) in enumerate(metric_names)
            arr = float.(species[name])
            if any(isnan, arr)
                println("NaN found in $name for species $sid in generation $gen")
            end
            if any(isinf, arr)
                println("Inf found in $name for species $sid in generation $gen")
            end
            for (jdx, fnc) in enumerate(agg_funcs)
                y = fnc(arr)
                if isnan(y)
                    println("NaN found after aggregation in $name for species $sid in generation $gen")
                elseif isinf(y)
                    println("Inf found after aggregation using $(agg_labels[jdx]) in $name for species $sid in generation $gen")
                elseif y > 1e6
                    data_over_time[idx, jdx, gi, species_idx] = NaN
                else
                    data_over_time[idx, jdx, gi, species_idx] = y
                end
            end
        end
    end
end

In [ ]:
# Aggregate across species: min of mins, median of medians, max of maxes
average_data_over_time = zeros(4, 3, length(generations))
for idx in 1:4
    average_data_over_time[idx, 1, :] = NaNMath.minimum.(eachslice(data_over_time[idx, 1, :, :]; dims=2))
    average_data_over_time[idx, 2, :] = nanmedian.(eachslice(data_over_time[idx, 2, :, :]; dims=2))
    average_data_over_time[idx, 3, :] = NaNMath.maximum.(eachslice(data_over_time[idx, 3, :, :]; dims=2))
end

# Plot 2x2 subplots
plots = []
for idx in 1:4
    p = plot(generations, average_data_over_time[idx, 1, :],
        yaxis=:log, label="min", title=metric_names[idx],
        xlabel="Generation", ylabel="Value", legend=:topleft)
    plot!(p, generations, average_data_over_time[idx, 2, :], label="median")
    plot!(p, generations, average_data_over_time[idx, 3, :], label="max")
    push!(plots, p)
end
plot(plots..., layout=(2, 2), size=(1000, 700))